### **FUENTES**:

PetFinder Kaggle:

https://www.kaggle.com/competitions/petfinder-adoption-prediction/data

First Tutorial:

https://towardsdatascience.com/how-to-train-an-image-classifier-in-pytorch-and-use-it-to-perform-basic-inference-on-single-images-99465a1e9bf5

Second Deep Tutorial:

https://rumn.medium.com/part-1-ultimate-guide-to-fine-tuning-in-pytorch-pre-trained-model-and-its-configuration-8990194b71e

Logo Recognition API:

https://heartbeat.comet.ml/logo-recognition-ios-application-using-machine-learning-and-flask-api-aec4eff3be11

Hybrid (multimodal) neural network architecture : Combination of tabular, textual and image inputs:

https://medium.com/@dave.cote.msc/hybrid-multimodal-neural-network-architecture-combination-of-tabular-textual-and-image-inputs-7460a4f82a2e



### **INDICACIONES PREVIAS**:

+ **Git**:
    + Clonamos el repo: `git clone "url_repo"`
    + Nos cambiamos a la rama main: `git checkout main`
    + (Opcional) Creamos una rama de trabajo desde main: `git checkout -b new-branch`

+ **Poetry**:
    + Instalamos poetry: https://python-poetry.org/docs/
    + Actualizamos dependencias: `poetry update`
    + Activamos el entorno:
        + Poetry 1.x: `poetry shell`
        + Poetry 2.x: `eval $(poetry env activate)` (o `poetry env activate` y copiar el comando que imprime)
    + Si VSCode no detecta el environment al correr una celda, cerrar y volver a abrir VSC

+ **Torch y CUDA**:
    + Este notebook requiere **torch >= 2.0** (usa la API `torch.amp`).
    + Ver la versión instalada: `poetry show torch`
    + Tabla de compatibilidad torch ↔ CUDA: https://pytorch.org/get-started/locally/ (versiones previas en https://pytorch.org/get-started/previous-versions/)
    + Instalar el CUDA Toolkit correspondiente: https://developer.nvidia.com/cuda-toolkit-archive
    + Verificar que CUDA esté funcional: correr en una celda `torch.cuda.is_available()`

In [1]:
import numpy as np
import pandas as pd
import random
import glob
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import os
import sys
import shutil
import time
import copy
import datetime
from tqdm import tqdm
#import matplotlib.pyplot as plt
#import seaborn as sns
#import cv2
#from PIL import Image
#from pathlib import Path

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
# API nueva (sin deprecation warnings). Antes: from torch.cuda.amp import autocast, GradScaler
from torch.amp import autocast, GradScaler
from torch.autograd import Variable
import torch.nn.functional as F

from joblib import load, dump

from utils import plot_confusion_matrix

# Appendeo el directorio de una carpeta arriba
nb_dir = os.path.dirname(os.path.abspath("05_Resnet50_3_train_augment.ipynb"))
dir = os.path.abspath(os.path.join(nb_dir,'..'))
sys.path.append(dir)

from augment.autoaugment import ImageNetPolicy	
from augment.cutout import Cutout

# Reproducibilidad: seedeo todo lo que genera aleatoriedad.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Verificamos que CUDA está funcional
torch.cuda.is_available()

c:\Users\User\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

**Seteo el Modelo**

Teoría de Resnet: https://towardsdatascience.com/introduction-to-resnets-c0a830a288a4

In [2]:
# Device global
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(num_classes=5, freeze_backbone=False):
    """Devuelve una ResNet50 preentrenada con cabeza nueva.
    Se llama de nuevo en cada trial de Optuna para evitar arrastrar pesos entre trials."""
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(device)


def build_optimizer(model, lr=1e-4, momentum=0.9, weight_decay=1e-3, head_lr_mult=10.0):
    """SGD con LR diferencial: backbone a `lr`, cabeza (fc) a `lr * head_lr_mult`.
    Defaults: lr=1e-4 para el backbone (ya sabe cosas útiles, solo ajuste fino)
              head_lr_mult=10 -> cabeza a 1e-3 (random, necesita aprender rápido)
              weight_decay=1e-3 (regularización más fuerte; antes 1e-4)."""
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if name.startswith("fc.") else backbone_params).append(p)
    param_groups = [
        {"params": backbone_params, "lr": lr},
        {"params": head_params,     "lr": lr * head_lr_mult},
    ]
    return optim.SGD(param_groups, momentum=momentum, weight_decay=weight_decay)

**Seteo parámetros, directorios y funciones**

In [3]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "dataset/train/train.csv")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "dataset/train_images")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

os.makedirs(PATH_TO_TEMP_FILES, exist_ok=True)
os.makedirs(PATH_TO_OPTUNA_ARTIFACTS, exist_ok=True)

MODEL_NAME = '04 ResNet Augment'
# 1.0.2: study fresco para correr de cero todo el pipeline (sin AutoAugment,
# Cutout chico, search space [5,12] epochs, TPE seed=42, MedianPruner, etc.).
# Mismo código que 1.0.1 — el bump es solo para no mezclarse con los trials
# previos en la sqlite y empezar limpio.
MODEL_VERSION = '1.0.2'

# Parametros y variables
CREATE_PYTORCH_DIRECTORIES = 1
BATCH_SIZE = 32                       # 32 da mejor throughput en GPU que 50
TEST_SIZE = 0.2
IMAGE_SIZE = 224                      # Tamaño nativo de ResNet50 (299 era de Inception)
# num_workers alto rompe DataLoader en Windows+Jupyter -> 0 en Windows, hasta 4 en Unix.
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 1)
USE_AMP = True                        # Mixed precision fp16 si hay GPU

# Armo el nuevo directorio de train
new_train_directory = os.path.join(BASE_DIR, 'work/train_images_classes')
os.makedirs(new_train_directory, exist_ok=True) # si ya existe el nombre, lo deja como está

# Armo el nuevo directorio de validación
new_val_directory = os.path.join(BASE_DIR, 'work/val_images_classes')
os.makedirs(new_val_directory, exist_ok=True)

# Definir las clases ordenadas
class_names = ['0', '1', '2', '3', '4']

# Mapear las etiquetas de las clases a números enteros consecutivos
class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

# Creo las carpetas de clases dentro de los directorios
for clase in class_names: # Una para cada clase
   os.makedirs(os.path.join(new_train_directory, str(clase)), exist_ok=True)
   os.makedirs(os.path.join(new_val_directory, str(clase)), exist_ok=True)


# Funciones para la carga y el preproceso
def resize_to_square(im):
    old_size = im.shape[:2] # old_size is in (height, width) format
    # Calcula el factor de escala necesario para redimensionar la imagen de manera que el lado más largo tenga el tamaño deseado 
    ratio = float(IMAGE_SIZE)/max(old_size)
    # Calcula las nuevas dimensiones de la imagen 
    new_size = tuple([int(x*ratio) for x in old_size])
    # Redimensiona la imagen con el nuevo tamaño
    im = cv2.resize(im, (new_size[1], new_size[0]))
    # Calcula las diferencias de tamaño y agrega pixeles (color negro) en los extremos para que quede centrada y cuadrada 
    delta_w = IMAGE_SIZE - new_size[1]
    delta_h = IMAGE_SIZE - new_size[0]
    top, bottom = delta_h//2, delta_h-(delta_h//2)
    left, right = delta_w//2, delta_w-(delta_w//2)
    color = [0, 0, 0]
    new_image = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT,value=color)
    return new_image


def load_image(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg') # Irá a la primera imagen de la mascota
    image = cv2.imread(path_to_image)
    # Convierte la imagen de BGR a RGB porque estos modelos esperan ese orden de canales
    image = cv2.convertScaleAbs(image)
    image= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    new_image = resize_to_square(image)
    return new_image


In [4]:

def visualize_pet(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg') # Irá a la primera imagen de la mascota
    # Cargar la imagen
    image_to_show = cv2.imread(path_to_image)
    # Convertir a formato RGB
    image_to_show = cv2.cvtColor(image_to_show, cv2.COLOR_BGR2RGB)
    # Visualizar la imagen
    plt.imshow(image_to_show)
    plt.axis('off')  # No mostrar los ejes
    plt.show()

def visualize_image(image):
    # Convierte la imagen a un formato de enteros (CV_8U)
    image = cv2.convertScaleAbs(image)
    image= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # Visualizar la imagen
    plt.imshow(image.astype(np.uint8))
    plt.axis('off')  # No mostrar los ejes
    plt.show()


**Cargo y Proceso Data**

Nota: Pytorch necesita que estén las imágenes en los distintos directorios según su clase y su participación en el training

In [5]:
# Cargo
train_df = pd.read_csv(PATH_TO_TRAIN)

# Split para validación (es por pet, no por imagen -> no hay leakage aun con varias fotos por pet)
train_data, val_data = train_test_split(train_df,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = train_df.AdoptionSpeed)




if CREATE_PYTORCH_DIRECTORIES == 0: # Poner en 0 si ya tengo las carpetas train_images_classes y val_images_classes con las imágenes copiadas
    def copy_imag(data, directorio_destino):
        """Copia TODAS las imágenes de cada pet (pet_id-1.jpg, pet_id-2.jpg, ...) a su carpeta de clase."""
        copied, skipped, missing = 0, 0, 0
        for _, row in data.iterrows():
            petID = row['PetID']
            adoption_speed = row['AdoptionSpeed']
            # Glob de todas las imágenes del pet, no sólo la primera.
            pattern = os.path.join(PATH_TO_IMAGES_DIR, f"{petID}-*.jpg")
            matches = glob.glob(pattern)
            if not matches:
                missing += 1
                continue
            for ruta_origen in matches:
                nombre_archivo = os.path.basename(ruta_origen)
                ruta_destino = os.path.join(directorio_destino, str(adoption_speed), nombre_archivo)
                if os.path.exists(ruta_destino):
                    skipped += 1
                    continue
                shutil.copy2(ruta_origen, ruta_destino)
                copied += 1
        print(f"Copia a {directorio_destino}: nuevas={copied}, ya existían={skipped}, pets sin imagen={missing}")

    # Copiar las imágenes al directorio de train
    copy_imag(train_data, new_train_directory)

    # Copiar las imágenes al directorio de val
    copy_imag(val_data, new_val_directory)

    print("Proceso completado.")

# Class weights para compensar desbalance de clases en AdoptionSpeed.
# Son a nivel pet: aproximación razonable porque el #imágenes por pet no
# varía demasiado entre clases.
class_weight_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=train_data['AdoptionSpeed'].values,
)
class_weights_t = torch.tensor(class_weight_arr, dtype=torch.float32, device=device)
print("Class weights:", class_weight_arr)

# Loss con class weights + label smoothing. El smoothing ayuda a QWK porque
# la métrica final penaliza menos errores a clases cercanas.
criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=0.05)

Class weights: [7.31341463 0.97038835 0.74266254 0.92013809 0.71456658]


In [6]:
# Genero los DataLoaders
def create_dataloaders(train_directory, val_directory, batch_size, num_workers):
    resize_to = int(IMAGE_SIZE * 256 / 224)  # ~256 cuando IMAGE_SIZE=224

    # Augmentation pipeline -- v1.0.1 (más liviano que v1.0.0):
    #   v1.0.0 tenía: RandomResizedCrop + HFlip + ColorJitter + AutoAugment + Cutout(IMAGE_SIZE//6)
    #   El stack con AutoAugment (políticas hechas para ImageNet: Solarize,
    #   Posterize, Equalize, etc.) sobre ColorJitter y Cutout grande resultó
    #   demasiado agresivo: el modelo no llegaba a converger en 10 epochs y el
    #   baseline bajó de 0.303 -> 0.267. Sacamos AutoAugment y achicamos Cutout.
    train_transforms = transforms.Compose([
        transforms.Resize(resize_to),                                # mantiene aspect ratio
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),  # crop aleatorio (augment)
        transforms.RandomHorizontalFlip(),                           # espejo horizontal (augment)
        transforms.ColorJitter(brightness=0.2, contrast=0.2,
                               saturation=0.2, hue=0.05),            # jitter de color (augment)
        # ImageNetPolicy() removido en 1.0.1 -> demasiado agresivo para PetFinder
        transforms.ToTensor(),
        Cutout(n_holes=1, length=IMAGE_SIZE // 8),                   # cutout chico (antes //6)
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Transformaciones de imagen para el conjunto de validación (sin data augment)
    val_transforms = transforms.Compose([
        transforms.Resize(resize_to),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Crear conjuntos de datos para el conjunto de entrenamiento y validación
    conjunto_entrenamiento = datasets.ImageFolder(train_directory, transform=train_transforms)
    conjunto_validacion = datasets.ImageFolder(val_directory, transform=val_transforms)

    # Asignar las clases ordenadas al conjunto de datos
    conjunto_entrenamiento.class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}
    conjunto_validacion.class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

    dl_kwargs = dict(
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
    )

    # Crear dataloaders para el conjunto de entrenamiento y validación
    train_dataloader = torch.utils.data.DataLoader(
        conjunto_entrenamiento, batch_size=batch_size, shuffle=True, **dl_kwargs
    )
    val_dataloader = torch.utils.data.DataLoader(
        conjunto_validacion, batch_size=batch_size, shuffle=False, **dl_kwargs
    )

    return train_dataloader, val_dataloader

# Aplico las funcion de los DataLoaders
train_dataloader, val_dataloader = create_dataloaders(new_train_directory , new_val_directory , BATCH_SIZE, NUM_WORKERS)

In [7]:
# Genero una lista de PetIDs con imagen en el orden en que aparecen en el data loader.
# os.path.basename maneja separadores tanto en Windows (\) como Unix (/).
test_sample_ids = [os.path.basename(s[0]).split('-')[0] for s in val_dataloader.dataset.samples]

**Entreno**

In [8]:
def train_val(model, criterion, dataloaders, datasets, device,
              num_epochs=20, lr=1e-4, momentum=0.9, weight_decay=1e-3,
              head_lr_mult=10.0, use_amp=True, grad_clip=1.0,
              patience=3, trial=None):

    # Optimizer con LR diferencial (backbone vs cabeza) y weight decay.
    optimizer = build_optimizer(model, lr=lr, momentum=momentum,
                                weight_decay=weight_decay, head_lr_mult=head_lr_mult)
    # Cosine annealing: LR alto al inicio, decae suavemente al final.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    # Mixed precision con la API nueva (torch.amp). enabled=False si no hay GPU -> no-op.
    scaler = GradScaler('cuda', enabled=use_amp and device.type == 'cuda')

    #Inicializo variables
    since = time.time()

    #Inicializo variable de mejor kappa entre trials
    try:
        #Intento obtener el mejor kappa de optuna
        previous_best = study.best_value
    except:
        #Si no hay, seteo -999
        previous_best = -999

    #Inicializo variables de mejor modelo y mejor accuracy y mejor kappa de este trial
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999
    best_epoch = -1   # para early stopping


    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        
        #Inicializo listas de kappa true y predicted y scores para esta epoch
        epoch_kappa_labels_true = []
        epoch_kappa_labels_predicted = []
        epoch_output_scores = []

        #Cada epoch tiene una fase de entrenamiento y validación
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            #Inicializo variables de loss y accuracy para esta fase de epoch
            epoch_phase_running_loss = 0.0
            epoch_phase_running_corrects = 0

            # Itero sobre los datos.
            for inputs, labels in tqdm(dataloaders[phase]):
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                # Zero the parameter gradients
                optimizer.zero_grad(set_to_none=True)

                # Forward (bajo autocast para mixed precision)
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    with autocast('cuda', enabled=use_amp and device.type == 'cuda'):
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        scaler.scale(loss).backward()
                        if grad_clip is not None:
                            # Unscale antes del clipping para clipear en magnitudes reales.
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        scaler.step(optimizer)
                        scaler.update()
                    elif phase == 'val':
                        #Agrego los valores de kappa true y predicted para cada batch en validación
                        epoch_kappa_labels_true.extend(labels.cpu().numpy().tolist())
                        epoch_kappa_labels_predicted.extend(preds.cpu().numpy().tolist())
                        outputs_np = outputs.detach().cpu().numpy()
                        epoch_output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                # Statistics for each phase
                epoch_phase_running_loss += loss.item() * inputs.size(0)
                epoch_phase_running_corrects += torch.sum(preds == labels.data)
                
                #END OF BATCH

            # Normalizo por # de IMÁGENES (no de pets). Con varias fotos por pet,
            # len(datasets[phase]) (= #pets) subestima el denominador.
            n_samples = len(dataloaders[phase].dataset)
            epoch_loss = epoch_phase_running_loss / n_samples
            epoch_acc = epoch_phase_running_corrects.double() / n_samples
            
            #Calculo el kappa para cada epoch
            if phase == 'train':
                #overall_train_losses.append(epoch_loss)
                current_kappa_score = np.nan
                img_kappa = np.nan
                pet_kappa = np.nan
                # Avanzo el scheduler una vez por epoch, después del train.
                scheduler.step()
            else:
                # Kappa a nivel imagen (cada foto es una muestra independiente).
                img_kappa = cohen_kappa_score(
                    epoch_kappa_labels_true, epoch_kappa_labels_predicted,
                    weights='quadratic'
                )
                # Kappa a nivel pet: promedio los logits de todas las fotos de
                # un mismo pet y hago argmax. Es la métrica "real" del problema
                # (la competencia evalúa por pet, no por foto).
                scores_arr = np.stack(epoch_output_scores, axis=0)
                score_cols = [f'score_{c}' for c in range(scores_arr.shape[1])]
                val_pred_df = pd.DataFrame(scores_arr, columns=score_cols)
                val_pred_df['PetID'] = test_sample_ids
                val_pred_df['true']  = epoch_kappa_labels_true
                pet_agg = val_pred_df.groupby('PetID', as_index=False).agg(
                    {'true': 'first', **{c: 'mean' for c in score_cols}}
                )
                pet_pred = pet_agg[score_cols].values.argmax(axis=1)
                pet_true = pet_agg['true'].values
                pet_kappa = cohen_kappa_score(pet_true, pet_pred, weights='quadratic')

                # Optimizo pet-level: es la métrica que importa.
                current_kappa_score = pet_kappa


            if phase == 'val':
                print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% '
                      f'ImgKappa: {img_kappa:.3f} PetKappa: {pet_kappa:.3f}')
            else:
                print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}%')

            # If this is the best Epoch so far -> Deep copy the model
            if phase == 'val' and current_kappa_score > best_kappa:
                best_acc = epoch_acc
                best_kappa = current_kappa_score
                best_model_wts = copy.deepcopy(model.state_dict())
                best_epoch = epoch


                #Best Epoch within a trial and better than previous trials
                if trial is not None and best_kappa > previous_best:

                    #Save test dataset with predictions
                    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                    predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                'pred':epoch_output_scores}).merge(val_data, on='PetID')
                    dump(predicted_df, predicted_filename)

                    #Generate and save CM (a nivel pet, que es como evaluamos)
                    cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                    plot_confusion_matrix(pet_true.tolist(), pet_pred.tolist()).write_image(cm_filename)

            #END OF PHASE

        # Optuna pruning: reporto el kappa de esta epoch al trial. Si el pruner
        # decide que este trial no vale la pena (vs. la mediana de trials previos
        # en la misma epoch), levanta TrialPruned y mata el trial AHORA, sin
        # esperar a las epochs restantes. Esto sí mata trials malos rápido.
        if trial is not None and not np.isnan(pet_kappa):
            trial.report(pet_kappa, epoch)
            if trial.should_prune():
                print(f'Trial {trial.number} pruned en epoch {epoch} (PetKappa={pet_kappa:.3f}).')
                raise optuna.TrialPruned()

        # Early stopping: corto si hace `patience` epochs que no mejora el PetKappa.
        # El best checkpoint ya está guardado en best_model_wts, así que no se pierde nada.
        if patience is not None and best_epoch >= 0 and (epoch - best_epoch) >= patience:
            print(f'Early stopping: {patience} epochs sin mejora '
                  f'(mejor epoch={best_epoch}, PetKappa={best_kappa:.3f}).')
            break

        #END OF EPOCH

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:.2f}%  Best val PetKappa: {:.3f}'.format(best_acc * 100, best_kappa))

    # Load best model weights
    model.load_state_dict(best_model_wts)

    # Save in optuna trial the best test dataset, cm and model weights
    if trial is not None and best_kappa > previous_best:
        # Desde optuna 4.0 los argumentos de upload_artifact son keyword-only.
        upload_artifact(study_or_trial=trial, file_path=predicted_filename, artifact_store=artifact_store)
        upload_artifact(study_or_trial=trial, file_path=cm_filename, artifact_store=artifact_store)

        file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
        model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
        # Guardo solo state_dict (más portable que torch.save del modelo entero).
        torch.save(model.state_dict(), model_path)
        upload_artifact(study_or_trial=trial, file_path=model_path, artifact_store=artifact_store)

    return model,best_kappa


# Modelo fresco para el baseline (antes se pasaba el global resnet50 y arrastraba pesos).
baseline_model = build_model(num_classes=len(class_names))

# HPs explícitos: lr=1e-4 backbone, cabeza a 1e-3 (head_lr_mult=10), weight_decay=1e-3.
# num_epochs=10 + patience=3 -> corre hasta converger y corta cuando plateaua.
best_model,_ = train_val(baseline_model, criterion,
                       dataloaders={'train': train_dataloader, 
                                    'val': val_dataloader}, 
                       datasets={'train': train_data, 'val': val_data}, 
                       device=device, 
                       num_epochs=10,
                       lr=1e-4,
                       head_lr_mult=10.0,
                       weight_decay=1e-3,
                       use_amp=USE_AMP,
                       patience=3)
# Guardo el modelo (state_dict, no el objeto completo)
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model.state_dict(), model_path)
print(f'Modelo guardado en {model_path}')

Epoch 0/9
----------


100%|██████████| 1459/1459 [05:50<00:00,  4.16it/s]


Train Loss: 1.6706 Acc: 27.41%


100%|██████████| 364/364 [00:36<00:00, 10.09it/s]


Val Loss: 1.6909 Acc: 28.70% ImgKappa: 0.204 PetKappa: 0.219
Epoch 1/9
----------


...

Val Loss: 1.6720 Acc: 30.59% ImgKappa: 0.253 PetKappa: 0.282
Epoch 7/9
----------


100%|██████████| 1459/1459 [05:53<00:00,  4.12it/s]


Train Loss: 1.6117 Acc: 34.41%


100%|██████████| 364/364 [00:34<00:00, 10.50it/s]


Val Loss: 1.6792 Acc: 29.95% ImgKappa: 0.246 PetKappa: 0.280
Early stopping: 3 epochs sin mejora (mejor epoch=4, PetKappa=0.283).
Training complete in 51m 36s
Best val Acc: 30.59%  Best val PetKappa: 0.283
Modelo guardado en ../work/optuna_temp_artifacts\04 ResNet Augment_1.0.2_20260428_164612.pth


In [9]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def optuna_train(trial):
    # CRÍTICO: modelo fresco por trial. Antes se reusaba `resnet50` global y cada
    # trial arrancaba desde los pesos del trial anterior -> HPO inválido.
    model = build_model(num_classes=len(class_names))

    epochs       = trial.suggest_int('epochs', 5, 12)
    lr           = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    momentum     = trial.suggest_float('momentum', 0.0, 0.95)
    # Rango extendido: el baseline usa 1e-3, así que dejo que Optuna explore
    # también regularización más fuerte (hasta 1e-2).
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
    head_lr_mult = trial.suggest_float('head_lr_mult', 1.0, 20.0)

    # patience=2 (más agresivo que el baseline=3): si dos epochs no mejoran,
    # corto. Combinado con el pruner de Optuna en train_val, los trials malos
    # mueren rápido y los buenos se entrenan hasta donde valga la pena.
    _, best_score = train_val(model, criterion,
                       dataloaders={'train': train_dataloader,
                                    'val': val_dataloader},
                       datasets={'train': train_data, 'val': val_data},
                       device=device,
                       num_epochs=epochs,
                       lr=lr,
                       momentum=momentum,
                       weight_decay=weight_decay,
                       head_lr_mult=head_lr_mult,
                       use_amp=USE_AMP,
                       patience=2,
                       trial=trial)


    return(best_score)

In [10]:
# Sampler reproducible: TPE con seed -> dada la misma DB, los trials siguientes
# se eligen de forma determinística. multivariate=True modela correlaciones
# entre HPs (lr y momentum, por ejemplo) en vez de tratarlos como independientes.
sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)

# Pruner: corta trials que en una epoch dada están por debajo de la mediana
# de los trials previos en esa misma epoch.
#  - n_startup_trials=2: los primeros 2 trials corren completos (necesitamos
#    datos para calcular la mediana).
#  - n_warmup_steps=2: dentro de un trial, no podo en las primeras 2 epochs
#    (hay que dar tiempo a que el modelo arranque).
#  - interval_steps=1: chequeo cada epoch.
pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=2, interval_steps=1)


def _log_trial(study, trial):
    """Callback: imprime el resultado de cada trial y el mejor hasta ahora."""
    state = trial.state.name
    value = trial.value if trial.value is not None else float('nan')
    print(f'>> Trial {trial.number} [{state}] kappa={value:.4f} | '
          f'best={study.best_value:.4f} (trial #{study.best_trial.number})')


study = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists=True,
                            sampler=sampler,
                            pruner=pruner)

# gc_after_trial=True libera memoria GPU entre trials (importante con ResNet50).
# callbacks=[_log_trial] da un resumen claro al final de cada trial.
study.optimize(optuna_train,
               n_trials=5,
               gc_after_trial=True,
               callbacks=[_log_trial])

# Resumen final del estudio
print('\n=== Resumen del estudio ===')
completed = [t for t in study.trials if t.state.name == 'COMPLETE']
pruned    = [t for t in study.trials if t.state.name == 'PRUNED']
failed    = [t for t in study.trials if t.state.name == 'FAIL']
print(f'Trials totales:    {len(study.trials)}  '
      f'(completos={len(completed)}, podados={len(pruned)}, fallidos={len(failed)})')
print(f'Best PetKappa:     {study.best_value:.4f}')
print(f'Best trial number: {study.best_trial.number}')
print(f'Best params:       {study.best_params}')

c:\Users\User\anaconda3\envs\ldi2_cuda\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-04-28 16:46:13,271] A new study created in RDB with name: 04 ResNet Augment_1.0.2


Epoch 0/6
----------


100%|██████████| 1459/1459 [06:21<00:00,  3.82it/s]


Train Loss: 1.6608 Acc: 29.44%


100%|██████████| 364/364 [00:36<00:00, 10.07it/s]


...

Val Loss: 1.7084 Acc: 27.69% ImgKappa: 0.135 PetKappa: 0.165
Epoch 2/8
----------


100%|██████████| 1459/1459 [06:06<00:00,  3.98it/s]


Train Loss: 1.6877 Acc: 27.50%


100%|██████████| 364/364 [00:38<00:00,  9.48it/s]
[I 2026-04-28 19:34:52,349] Trial 4 pruned. 


Val Loss: 1.7044 Acc: 27.72% ImgKappa: 0.152 PetKappa: 0.172
Trial 4 pruned en epoch 2 (PetKappa=0.172).
>> Trial 4 [PRUNED] kappa=0.1715 | best=0.3235 (trial #0)

=== Resumen del estudio ===
Trials totales:    5  (completos=3, podados=2, fallidos=0)
Best PetKappa:     0.3235
Best trial number: 0
Best params:       {'epochs': 7, 'lr': 0.06351221010640701, 'momentum': 0.6953942447208348, 'weight_decay': 0.0006251373574521745, 'head_lr_mult': 3.964354168406294}
